In [1]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import BertModel, BertTokenizer, get_scheduler, set_seed, AutoTokenizer, AutoModel

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Train.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_train = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_train = df_train.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Test.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_test = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_test = df_test.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Dev.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_val = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_val = df_val.drop(['id'], axis=1)

In [3]:
import re
import torch

# Define the mappings
aspect_to_index = {'ambience#general': 0, 'drinks#prices': 1, 'drinks#quality': 2, 'drinks#style&options': 3, 'food#prices': 4,
                   'food#quality': 5, 'food#style&options': 6, 'location#general': 7, 'restaurant#general': 8, 'restaurant#miscellaneous': 9,
                   'restaurant#prices': 10, 'service#general': 11}
# Adding 'None' to handle unspecified sentiments for any detected aspect
sentiment_to_index = {'positive': 1, 'negative': 0, 'neutral': 0, 'none': 0}

# Preprocess the labels
def preprocess_labels(label_str):
    labels = re.findall(r'\{(.*?)\}', label_str.lower())
    aspects = torch.zeros(len(aspect_to_index), dtype=torch.long)
    sentiments = torch.full((len(aspect_to_index),), sentiment_to_index['none'], dtype=torch.long)

    # Initialize with 'None'
    for label in labels:
        if ', ' in label:
            aspect, sentiment = label.split(', ')
            if aspect in aspect_to_index and sentiment in sentiment_to_index:  # Only process known aspects with sentiments
                idx = aspect_to_index[aspect]
                aspects[idx] = 1
                sentiments[idx] = sentiment_to_index[sentiment]

    return aspects, sentiments

df_train['aspects'], df_train['sentiments'] = zip(*df_train['label'].apply(preprocess_labels))
df_test['aspects'], df_test['sentiments'] = zip(*df_test['label'].apply(preprocess_labels))
df_val['aspects'], df_val['sentiments'] = zip(*df_val['label'].apply(preprocess_labels))

In [ ]:
from bs4 import BeautifulSoup

for df in [df_train, df_test, df_val]:
  for sentence in range(0, len(df.text)):
    # xóa tag, link http
    processed_feature = BeautifulSoup(str(df.text[sentence]), "html.parser").get_text()
    #email-id
    processed_feature = re.sub('b[w-]+?@w+?.w{2,4}b', 'emailadd', processed_feature)
    #url
    processed_feature = re.sub('(http[s]?S+)|(w+.[A-Za-z]{2,4}S*)', 'urladd', processed_feature)

    # Xóa tất cả các ký tự đặc biệt
    processed_feature = re.sub(r'\W', ' ', processed_feature)

    # xóa từ có chứa chữ số
    # processed_feature = re.sub('W*dw*', '', processed_feature)

    # Chuyển đổi sang chữ thường
    processed_feature = processed_feature.lower()

    # Thay thế nhiều khoảng trắng bằng một khoảng trắng
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # processed_features.append(processed_feature)
    df.text[sentence] = processed_feature

In [5]:
df_train['aspects'] = df_train['aspects'].apply(lambda x: x.numpy())
df_train['sentiments'] = df_train['sentiments'].apply(lambda x: x.numpy())

df_test['aspects'] = df_test['aspects'].apply(lambda x: x.numpy())
df_test['sentiments'] = df_test['sentiments'].apply(lambda x: x.numpy())

df_val['aspects'] = df_val['aspects'].apply(lambda x: x.numpy())
df_val['sentiments'] = df_val['sentiments'].apply(lambda x: x.numpy())

In [6]:
df_train

,text,label,aspects,sentiments
0,giá 53k size vừa,"{DRINKS#PRICES, neutral}, {DRINKS#STYLE&OPTION...","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,nhưng nói chung cũng hơi đắt,"{RESTAURANT#PRICES, negative}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,mình ăn rất hôi mùi dầu,"{FOOD#QUALITY, negative}","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,mình ăn chưa baoh thấy mùi hôi hải sản,"{FOOD#QUALITY, positive}","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
4,3 dĩa vs 2 lon revive mà có 190k thui,"{RESTAURANT#PRICES, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
...,...,...,...,...
7023,cơ mà hôm đó bạn nv order có vẻ khó chịu với m...,"{SERVICE#GENERAL, negative}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7024,tổng hoá đơn phần bò sốt là 115k,"{FOOD#PRICES, neutral}","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7025,quán trang trí bắt mắt và sáng sủa,"{AMBIENCE#GENERAL, positive}","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7026,trà đào ở đây đậm đà hơn ngon hơn mấy chỗ khác...,"{DRINKS#QUALITY, positive}, {RESTAURANT#MISCEL...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [98]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [107]:
MAX_LEN = 128
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 5e-5
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [99]:
class CustomDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.text
        self.targets = self.data.aspects
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [100]:
training_set = CustomDataset(df_train, tokenizer, 128)
testing_set = CustomDataset(df_test, tokenizer, 128)
validation_set = CustomDataset(df_val, tokenizer, 128)

In [101]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [110]:
class AspectClass(torch.nn.Module):
    def __init__(self):
        super(AspectClass, self).__init__()
        self.l1 = transformers.BertModel.from_pretrained('bert-base-uncased')
        self.l2 = torch.nn.Dropout(0.3)
        self.l3 = torch.nn.Linear(768, 12)

    def forward(self, ids, mask, token_type_ids):
        _, output_1= self.l1(ids, attention_mask = mask, token_type_ids = token_type_ids, return_dict=False)
        output_2 = self.l2(output_1)
        output = self.l3(output_2)
        return output

modelAspect = AspectClass()
modelAspect.to(device)

BERTClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affin

In [111]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [112]:
optimizer = torch.optim.AdamW(params =  model.parameters(), lr=LEARNING_RATE)

In [113]:
def train(epoch):
    modelAspect.train()
    for _,data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = modelAspect(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [114]:
for epoch in range(EPOCHS):
    train(epoch)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Epoch: 0, Loss:  0.72756427526474
Epoch: 1, Loss:  0.24592749774456024
Epoch: 2, Loss:  0.13428044319152832
Epoch: 3, Loss:  0.1021900624036789
Epoch: 4, Loss:  0.0914035439491272
Epoch: 5, Loss:  0.07251015305519104
Epoch: 6, Loss:  0.04998811334371567
Epoch: 7, Loss:  0.03792431205511093
Epoch: 8, Loss:  0.04660607874393463
Epoch: 9, Loss:  0.011887445114552975


In [115]:
def validation(epoch):
    modelAspect.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for _, data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = modelAspect(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
    return fin_outputs, fin_targets

In [117]:
from sklearn import metrics

# for epoch in range(EPOCHS):
outputs, targets = validation(epoch)
outputs = np.array(outputs) >= 0.5
accuracy = metrics.accuracy_score(targets, outputs)
f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
precision_score = metrics.precision_score(targets, outputs, average='micro')
recall_score = metrics.recall_score(targets, outputs, average='micro')
print(f"Accuracy Score = {accuracy}")
print(f"F1 Score (Micro) = {f1_score_micro}")
print(f"Precision = {precision_score}")
print(f"Recall = {recall_score}")

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Accuracy Score = 0.6233230134158927
F1 Score (Micro) = 0.787523629489603
Precision = 0.7827884254039834
Recall = 0.7923164701407379


In [129]:
class CustomDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.text
        self.targets = self.data.sentiments
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [130]:
training_set = CustomDataset(df_train, tokenizer, 128)
testing_set = CustomDataset(df_test, tokenizer, 128)
validation_set = CustomDataset(df_val, tokenizer, 128)

In [131]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [132]:
class SentimentClass(torch.nn.Module):
    def __init__(self):
        super(SentimentClass, self).__init__()
        self.l1 = transformers.BertModel.from_pretrained('bert-base-uncased')
        self.l2 = torch.nn.Dropout(0.3)
        self.l3 = torch.nn.Linear(768, 12)

    def forward(self, ids, mask, token_type_ids):
        _, output_1= self.l1(ids, attention_mask = mask, token_type_ids = token_type_ids, return_dict=False)
        output_2 = self.l2(output_1)
        output = self.l3(output_2)
        return output

modelSentiment = SentimentClass()
modelSentiment.to(device)

SentimentClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

In [133]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [134]:
optimizer = torch.optim.AdamW(params =  modelSentiment.parameters(), lr=LEARNING_RATE)

In [135]:
def train(epoch):
    modelSentiment.train()
    for _,data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = modelSentiment(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [136]:
for epoch in range(EPOCHS):
    train(epoch)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Epoch: 0, Loss:  0.780042827129364
Epoch: 1, Loss:  0.23984120786190033
Epoch: 2, Loss:  0.14880946278572083
Epoch: 3, Loss:  0.12017863988876343
Epoch: 4, Loss:  0.07215334475040436
Epoch: 5, Loss:  0.044922344386577606
Epoch: 6, Loss:  0.036727674305438995
Epoch: 7, Loss:  0.03256458044052124
Epoch: 8, Loss:  0.0595591738820076
Epoch: 9, Loss:  0.042754851281642914


In [137]:
def validation(epoch):
    modelSentiment.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for _, data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = modelSentiment(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
    return fin_outputs, fin_targets

In [138]:
from sklearn import metrics

# for epoch in range(EPOCHS):
outputs, targets = validation(epoch)
outputs = np.array(outputs) >= 0.5
accuracy = metrics.accuracy_score(targets, outputs)
f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
precision_score = metrics.precision_score(targets, outputs, average='micro')
recall_score = metrics.recall_score(targets, outputs, average='micro')
print(f"Accuracy Score = {accuracy}")
print(f"F1 Score (Micro) = {f1_score_micro}")
print(f"Precision = {precision_score}")
print(f"Recall = {recall_score}")

Accuracy Score = 0.6444788441692466
F1 Score (Micro) = 0.6754482253933407
Precision = 0.7504065040650406
Recall = 0.614105123087159
